# Managing Windows Updates and Ensuring Server Auto-Restart

This notebook addresses two important concerns when using Windows 11 Home as a development server:

1. **Controlling Windows Update behavior** to minimize unexpected restarts
2. **Configuring automatic server restarts** after system reboots

While Windows 11 Home doesn't allow completely disabling updates (unlike Pro/Enterprise editions), we can still implement strategies to better manage them and ensure our development environment recovers automatically after reboots.


## 1. Managing Windows Update Behavior

### Checking Current Windows Update Settings

First, let's check your current Windows Update settings using PowerShell commands. These commands will help you understand your current configuration.

In [ ]:
# Check current Windows Update settings
Get-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "ActiveHoursStart" -ErrorAction SilentlyContinue
Get-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "ActiveHoursEnd" -ErrorAction SilentlyContinue

# Check if updates are currently paused
$updateSettings = Get-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -ErrorAction SilentlyContinue
if ($updateSettings.PausedFeatureStatus -eq 1) {
    Write-Host "Feature updates are paused until: $($updateSettings.PausedFeatureDate)"
} else {
    Write-Host "Feature updates are not paused"
}

if ($updateSettings.PausedQualityStatus -eq 1) {
    Write-Host "Quality updates are paused until: $($updateSettings.PausedQualityDate)"
} else {
    Write-Host "Quality updates are not paused"
}

### Options for Managing Windows Update Restarts

While Windows 11 Home doesn't allow completely disabling updates, here are the most effective approaches you can use:

#### Option 1: Configure Active Hours

Setting active hours tells Windows not to restart during times when you're actively using your computer (up to an 18-hour window).

In [ ]:
# Set active hours (adjust to your working hours)
# Note: Windows 11 allows up to 18 hours between start and end
# This example sets active hours from 8:00 AM to 2:00 AM (18 hours)
$startHour = 8  # 8:00 AM
$endHour = 2    # 2:00 AM next day

try {
    # You need to run PowerShell as Administrator for this to work
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "ActiveHoursStart" -Value $startHour -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "ActiveHoursEnd" -Value $endHour -Type DWord
    Write-Host "Successfully set active hours from $startHour:00 to $endHour:00"
} catch {
    Write-Host "Error setting active hours. Make sure you're running PowerShell as Administrator."
    Write-Host "You can also set this via Windows Settings > Windows Update > Advanced options > Active hours"
}

#### Option 2: Pause Updates Temporarily

You can pause updates for a limited time (up to 35 days in Windows 11 Home). This is useful when you need guaranteed uptime for a specific period.

In [ ]:
# Pause updates for the maximum allowed time (usually 5 weeks/35 days)
# Note: This must be run as Administrator to work
try {
    # Calculate the date 35 days from today
    $pauseUntil = (Get-Date).AddDays(35).ToString("yyyy-MM-dd")
    
    # Set registry keys to pause updates
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "PausedFeatureStatus" -Value 1 -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "PausedFeatureDate" -Value $pauseUntil
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "PausedQualityStatus" -Value 1 -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "PausedQualityDate" -Value $pauseUntil
    
    Write-Host "Updates paused until $pauseUntil"
} catch {
    Write-Host "Error pausing updates. Make sure you're running PowerShell as Administrator."
    Write-Host "You can also pause updates via Windows Settings > Windows Update > Pause updates"
}

#### Option 3: Configure Update Notification Settings

You can modify the Registry to get more notifications before updates. While this doesn't prevent updates, it gives you more warning time.

In [ ]:
# Configure Windows Update to notify before downloading and installing
# Note: Must run as Administrator
try {
    # This increases the notification window for pending restarts
    New-Item -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate" -Force | Out-Null
    New-Item -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate\AU" -Force | Out-Null
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate\AU" -Name "AUOptions" -Value 2 -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate" -Name "SetAutoRestartNotificationDisable" -Value 0 -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate" -Name "NoAutoRebootWithLoggedOnUsers" -Value 1 -Type DWord
    
    Write-Host "Update notification settings applied successfully"
} catch {
    Write-Host "Error configuring update settings. Make sure you're running PowerShell as Administrator."
}

#### Option 4: Use the Group Policy Editor (not available in Windows 11 Home)

For reference only: Windows 11 Pro/Enterprise/Education users have access to the Group Policy Editor which provides more control over update behavior. Home edition users would need to upgrade to gain access to these features.

#### Recommendation

The most reliable approach is to **combine options 1 and 2** - set appropriate active hours AND pause updates during critical periods when your server absolutely cannot be interrupted.

## 2. Ensuring Automatic Server Restart After Reboot

Now that we've managed Windows updates as best as possible, let's set up automatic restart for your services.

### Creating Server Start Scripts

First, let's create Python scripts that can start your frontend and backend services. These scripts will be called automatically when Windows starts up.

In [ ]:
# File: start_servers.py
# This script will start both frontend and backend servers

import os
import subprocess
import logging
import time
from datetime import datetime
import sys
from pathlib import Path

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "server_autostart.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def start_servers():
    """Start both frontend and backend servers using Docker Compose"""
    try:
        # Record startup attempt
        logging.info(f"Starting servers at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Change to project directory
        project_dir = "C:/Users/USER/Documents/workspace/partnersclub/simulation"
        os.chdir(project_dir)
        logging.info(f"Changed directory to {project_dir}")
        
        # Start Docker containers if not already running
        try:
            result = subprocess.run(
                ["docker", "ps", "--filter", "name=simulation", "--format", "{{.Names}}"],
                capture_output=True,
                text=True,
                check=True
            )
            
            if "simulation" not in result.stdout:
                logging.info("Docker containers not running. Starting them now...")
                subprocess.run(
                    ["docker", "compose", "up", "-d"],
                    check=True
                )
                logging.info("Docker containers started successfully")
            else:
                logging.info("Docker containers already running")
        except subprocess.CalledProcessError as e:
            logging.error(f"Error checking or starting Docker containers: {e}")
            logging.error(f"Command output: {e.output}")
            raise
            
        # Wait for services to fully start
        time.sleep(10)
        
        logging.info("Servers started successfully")
        return True
    
    except Exception as e:
        logging.error(f"Error starting servers: {e}")
        return False

if __name__ == "__main__":
    success = start_servers()
    sys.exit(0 if success else 1)

### Creating a Windows Task to Run the Script at Startup

Now, let's create a scheduled task that runs our script when Windows starts up. We'll use the Windows Task Scheduler via a Python script.

In [ ]:
# File: setup_autostart_task.py
# This script creates a scheduled task to start our servers when Windows boots

import os
import subprocess
import sys
from pathlib import Path

def create_startup_task():
    """Create a Windows scheduled task to run our start script at system startup"""
    # Get the current directory where our scripts are located
    script_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation")
    start_script = script_dir / "start_servers.py"
    
    # Create the script if it doesn't exist
    if not start_script.exists():
        with open(start_script, "w") as f:
            f.write("""
import os
import subprocess
import logging
import time
from datetime import datetime
import sys
from pathlib import Path

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "server_autostart.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def start_servers():
    \"\"\"Start both frontend and backend servers using Docker Compose\"\"\"
    try:
        # Record startup attempt
        logging.info(f"Starting servers at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Change to project directory
        project_dir = "C:/Users/USER/Documents/workspace/partnersclub/simulation"
        os.chdir(project_dir)
        logging.info(f"Changed directory to {project_dir}")
        
        # Start Docker containers if not already running
        try:
            result = subprocess.run(
                ["docker", "ps", "--filter", "name=simulation", "--format", "{{.Names}}"],
                capture_output=True,
                text=True,
                check=True
            )
            
            if "simulation" not in result.stdout:
                logging.info("Docker containers not running. Starting them now...")
                subprocess.run(
                    ["docker", "compose", "up", "-d"],
                    check=True
                )
                logging.info("Docker containers started successfully")
            else:
                logging.info("Docker containers already running")
        except subprocess.CalledProcessError as e:
            logging.error(f"Error checking or starting Docker containers: {e}")
            logging.error(f"Command output: {e.output}")
            raise
            
        # Wait for services to fully start
        time.sleep(10)
        
        logging.info("Servers started successfully")
        return True
    
    except Exception as e:
        logging.error(f"Error starting servers: {e}")
        return False

if __name__ == "__main__":
    success = start_servers()
    sys.exit(0 if success else 1)
""")
    
    # Create the task using schtasks.exe
    task_name = "SimulationServersAutoStart"
    python_path = sys.executable
    command = f"{python_path} \"{start_script}\""
    
    # Create task that runs at system startup with highest privileges
    cmd = [
        "schtasks", "/create", "/tn", task_name, "/tr", command,
        "/sc", "onstart", "/ru", "SYSTEM", "/f"  # Force overwrite if exists
    ]
    
    try:
        subprocess.run(cmd, check=True)
        print(f"Successfully created startup task '{task_name}'")
        print(f"Script will run: {command}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error creating task: {e}")
        print("You may need to run this script as Administrator")
        return False

if __name__ == "__main__":
    success = create_startup_task()
    sys.exit(0 if success else 1)

### Testing the Autostart Configuration

To test if our autostart configuration works:

1. Run the `setup_autostart_task.py` script as Administrator to create the scheduled task
2. Reboot your computer
3. Check the log file at `C:/Users/USER/Documents/workspace/partnersclub/simulation/logs/server_autostart.log`

You can also manually run the start script to test it without rebooting.

Let's create a script to check the status of our servers:

In [ ]:
# File: check_server_status.py
# This script checks if our servers are running properly

import subprocess
import requests
import logging
import sys
from pathlib import Path

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "server_status.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def check_docker_containers():
    """Check if the Docker containers are running"""
    try:
        result = subprocess.run(
            ["docker", "ps", "--format", "table {{.Names}}\t{{.Status}}"],
            capture_output=True,
            text=True,
            check=True
        )
        logging.info("Docker containers status:")
        for line in result.stdout.splitlines():
            logging.info(line)
        
        # Check specifically for our simulation containers
        containers = subprocess.run(
            ["docker", "ps", "--filter", "name=simulation", "--format", "{{.Names}}"],
            capture_output=True,
            text=True,
            check=True
        )
        
        if "simulation" in containers.stdout:
            logging.info("✅ Simulation containers are running")
            return True
        else:
            logging.warning("❌ Simulation containers are not running")
            return False
    
    except subprocess.CalledProcessError as e:
        logging.error(f"Error checking Docker status: {e}")
        return False

def check_api_health():
    """Check if our backend API is responding"""
    try:
        # Adjust this URL to match your backend health endpoint
        response = requests.get("http://localhost:8000/health", timeout=5)
        if response.status_code == 200:
            logging.info(f"✅ Backend API is healthy: {response.json()}")
            return True
        else:
            logging.warning(f"❌ Backend API returned status code {response.status_code}")
            return False
    except Exception as e:
        logging.error(f"❌ Error checking backend API: {e}")
        return False

def check_frontend():
    """Check if frontend is serving"""
    try:
        # Adjust this URL to match your frontend
        response = requests.get("http://localhost:5173", timeout=5)
        if response.status_code == 200:
            logging.info("✅ Frontend is serving content")
            return True
        else:
            logging.warning(f"❌ Frontend returned status code {response.status_code}")
            return False
    except Exception as e:
        logging.error(f"❌ Error checking frontend: {e}")
        return False

if __name__ == "__main__":
    logging.info("Starting server status check...")
    
    docker_ok = check_docker_containers()
    api_ok = check_api_health()
    frontend_ok = check_frontend()
    
    if all([docker_ok, api_ok, frontend_ok]):
        logging.info("✅ All systems operational")
        sys.exit(0)
    else:
        logging.warning("❌ Some services are not running correctly")
        sys.exit(1)

### Creating a System Uptime Monitor

It's also useful to monitor your system's uptime so you can track when unexpected restarts happen. Let's create a script that logs system uptime information:

In [ ]:
# File: monitor_uptime.py
# This script monitors and logs system uptime information

import subprocess
import psutil
import datetime
import time
import logging
from pathlib import Path
import sys

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "uptime_monitor.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def get_system_uptime():
    """Get the current system uptime"""
    boot_time = datetime.datetime.fromtimestamp(psutil.boot_time())
    now = datetime.datetime.now()
    uptime = now - boot_time
    
    # Format as days, hours, minutes, seconds
    days, remainder = divmod(uptime.total_seconds(), 86400)
    hours, remainder = divmod(remainder, 3600)
    minutes, seconds = divmod(remainder, 60)
    
    uptime_str = f"{int(days)}d {int(hours)}h {int(minutes)}m {int(seconds)}s"
    boot_time_str = boot_time.strftime('%Y-%m-%d %H:%M:%S')
    
    return uptime_str, boot_time_str

def check_pending_updates():
    """Check if there are pending Windows updates"""
    try:
        result = subprocess.run(
            ["powershell", "-Command", "Get-WULastScanSuccessDate"],
            capture_output=True,
            text=True,
            check=True
        )
        last_scan = result.stdout.strip()
        
        result = subprocess.run(
            ["powershell", "-Command", "Get-WindowsUpdate"],
            capture_output=True,
            text=True,
            check=True
        )
        updates = result.stdout.strip()
        
        if "No updates found" in updates:
            return "No pending updates"
        else:
            return f"Pending updates found. Last scan: {last_scan}"
    except:
        return "Unable to check for Windows updates"

def log_system_info():
    """Log system uptime and related information"""
    uptime_str, boot_time_str = get_system_uptime()
    logging.info(f"System boot time: {boot_time_str}")
    logging.info(f"Current uptime: {uptime_str}")
    
    # Check memory usage
    mem = psutil.virtual_memory()
    logging.info(f"Memory usage: {mem.percent}% (Used: {mem.used/1024/1024/1024:.1f} GB, "
                f"Available: {mem.available/1024/1024/1024:.1f} GB)")
    
    # Check CPU usage
    cpu_percent = psutil.cpu_percent(interval=1)
    logging.info(f"CPU usage: {cpu_percent}%")
    
    # Check pending Windows updates
    update_status = check_pending_updates()
    logging.info(f"Windows Update status: {update_status}")

if __name__ == "__main__":
    if "--daemon" in sys.argv:
        # Run as a daemon and log every hour
        logging.info("Starting uptime monitor daemon")
        try:
            while True:
                log_system_info()
                time.sleep(3600)  # Sleep for 1 hour
        except KeyboardInterrupt:
            logging.info("Uptime monitor stopped")
    else:
        # Run once
        log_system_info()

### Setting Up a Windows Task for Regular Uptime Monitoring

Let's create a scheduled task to run our uptime monitor regularly:

In [ ]:
# File: setup_monitoring_task.py
# This script creates a scheduled task for regular system monitoring

import subprocess
import sys
from pathlib import Path

def create_monitoring_task():
    """Create a Windows scheduled task for regular monitoring"""
    script_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation")
    monitor_script = script_dir / "monitor_uptime.py"
    
    task_name = "SimulationSystemMonitor"
    python_path = sys.executable
    command = f"{python_path} \"{monitor_script}\""
    
    # Create task that runs hourly
    cmd = [
        "schtasks", "/create", "/tn", task_name, "/tr", command,
        "/sc", "hourly", "/ru", "SYSTEM", "/f"  # Force overwrite if exists
    ]
    
    try:
        subprocess.run(cmd, check=True)
        print(f"Successfully created monitoring task '{task_name}'")
        print(f"Script will run: {command}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error creating task: {e}")
        print("You may need to run this script as Administrator")
        return False

if __name__ == "__main__":
    success = create_monitoring_task()
    sys.exit(0 if success else 1)

## Summary and Recommendations

### Managing Windows Updates

1. **Set Active Hours** (8:00 AM to 2:00 AM) to prevent updates during your working hours
2. **Pause Updates** during critical periods when you need guaranteed uptime
3. **Configure Update Notifications** to get more warning before restarts

### Server Auto-Restart

1. **Create the `start_servers.py` script** to automatically start your Docker containers
2. **Run `setup_autostart_task.py` as Administrator** to create the scheduled task
3. **Test the setup** by manually running the script or rebooting your system

### Monitoring

1. **Use `check_server_status.py`** to verify if all services are running correctly
2. **Set up regular monitoring** with `setup_monitoring_task.py` to track system uptime

By implementing these solutions, you'll minimize the impact of Windows updates and ensure your services restart automatically after any system reboots.